In [7]:
%pip install fastapi uvicorn pydantic


[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import io
import joblib
import matplotlib.pyplot as plt
import uvicorn
import numpy as np
import logging
from pathlib import Path as FSPath 
from typing import List
from joblib import load
import nest_asyncio
from enum import Enum
from fastapi import FastAPI,Response, HTTPException, Path as Param
from fastapi.responses import StreamingResponse
from fastapi.responses import JSONResponse
from pydantic import BaseModel,  conlist, Field
from sklearn.pipeline import Pipeline

In [40]:
# Directorio donde están tus modelos
MODELS_DIR = Path.cwd().parent  / "models"

# Diccionario para alojar pipelines por número de clusters
pipelines: dict[int, joblib.load] = {}

In [83]:
app = FastAPI(title='Implementando un modelo de Machine Learning')

# pipelines[k] = {"model": Pipeline, "X_pca": np.ndarray}
pipelines: dict[int, dict] = {}

@app.get("/")
async def root():
    return {
        "message": "Service up ✅",
        "endpoints": {
            "list_pipelines": "/pipelines/",
            "plot_cluster": "/plot/{k}",
            "docs": "/docs"
        }
    }


In [85]:
@app.on_event("startup")
def load_data_and_pipelines():
    for path in MODELS_DIR.glob("pipeline_kmeans*.joblib"):
        m = re.match(r"^pipeline_kmeans(\d+)$", path.stem)
        if not m:
            logger.warning("Ignorando nombre inesperado: %s", path.name)
            continue

        k = int(m.group(1))
        try:
            pipelines[k] = joblib.load(path)
            PIPELINE_LOAD_SUCCESS.inc()
            logger.info("Cargado pipeline k=%d", k)
        except Exception as err:
            PIPELINE_LOAD_FAILURE.inc()
            logger.error("Error cargando %s: %s", path.name, err, exc_info=True)


C:\Users\crist\AppData\Local\Temp\ipykernel_16284\2874915712.py:1: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")


### Endpoint para listar pipelines disponibles

In [86]:
@app.get("/pipelines", summary="Listar modelos K-means disponibles")
async def list_pipelines():
    lista = [{"k": k, "filename": f"pipeline_kmeans{k}.joblib"} 
             for k in sorted(pipelines.keys())]
    return JSONResponse(content={"pipelines": lista})


### Endpoint /plot/{k} para visualizar clusters

In [88]:
# 3. Endpoint para generar el plot usando el pipeline ya cargado
from fastapi import FastAPI, HTTPException
from fastapi.responses import StreamingResponse
from io import BytesIO
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

app = FastAPI()

# pipelines ya fue poblado en startup: { k: {"model": Pipeline, "X_pca": np.ndarray}, ... }

@app.get("/plot/{k}")
def plot_clusters(k: int):
    entry = pipelines.get(k)
    if entry is None:
        raise HTTPException(status_code=404, detail=f"No existe pipeline para k={k}")

    # Extraer modelo y datos
    kmeans_pipeline = entry["model"]
    X_pca = entry["X_pca"]

    # Predecir etiquetas de cluster
    try:
        labels = kmeans_pipeline.predict(X_pca)
    except Exception as err:
        raise HTTPException(status_code=500, detail=f"Error en predicción: {err}")

    # Construir la figura
    fig, ax = plt.subplots(figsize=(6, 6))
    scatter = ax.scatter(
        X_pca[:, 0],
        X_pca[:, 1],
        c=labels,
        cmap="viridis",
        alpha=0.7
    )
    ax.set_title(f"KMeans k={k}")
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

    # Serializar a PNG
    buf = BytesIO()
    fig.savefig(buf, format="png", dpi=100, bbox_inches="tight")
    plt.close(fig)
    buf.seek(0)

    return StreamingResponse(buf, media_type="image/png")

2025-08-25 16:48:38,287 ERROR: Task exception was never retrieved
future: <Task finished name='Task-172' coro=<Server.serve() done, defined at d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\server.py:69> exception=KeyboardInterrupt()>
Traceback (most recent call last):
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\main.py", line 580, in run
    server.run()
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.venv\lib\site-packages\uvicorn\server.py", line 67, in run
    return asyncio.run(self.serve(sockets=sockets))
  File "d:\crist\OneDrive\Estudios\Mg Data Science_2024\Trimestre 5\Desarrollo de proyectos y productos de datos\Tareas\SegmentacionClientes\.

¡Corriendo la celda que viene echaremos a andar el servidor!

Esto causará que el notebook se bloquee (No podremos correr más celdas) hasta que interrumpamos de forma manual el kernel.
Podemos hacer eso haciendo click en la pestaña **kernel** y luego **Interrupt**.

In [89]:
logging.basicConfig(level=logging.DEBUG)

# Esto deja correr al servidor en un ambiente interactivo como un jupyter notebook
nest_asyncio.apply()

# Donde se hospedará el servidor
host = "127.0.0.1"

# Iniciamos el servidor
uvicorn.run(app, host= host, port=8000)

INFO:     Started server process [16284]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://127.0.0.1:8000 (Press CTRL+C to quit)


INFO:     127.0.0.1:65270 - "GET /plot/3 HTTP/1.1" 200 OK


INFO:     Shutting down
INFO:     Waiting for application shutdown.
INFO:     Application shutdown complete.
INFO:     Finished server process [16284]


¡El servidor está corriendo! Vamos a http://localhost:8000/ para verlo en acción.